# Drone tracking experiments

Compare full-frame detection, trajectory crops, and consensus-assisted tracking on
recorded video. Each experiment uses the project's existing pipeline. Labeled clips
also get per-video and aggregate MOT scores.

From the repository root, launch JupyterLab with `uv run jupyter lab notebooks/pipeline_experiments.ipynb`.
Select **Python (Drone Tracking v2)** in Jupyter, edit the configuration below, and
run the notebook from the top. Select one clip or one preset for a shorter first run.
The selected practice videos can take several minutes at 1280 px.

In [ ]:
import json
import re
import shlex
import subprocess
import sys
import tempfile
from dataclasses import asdict, dataclass, field, replace
from datetime import datetime
from pathlib import Path
from time import perf_counter

import cv2
import motmetrics as mm
import numpy as np
import pandas as pd
from IPython.display import Image, display

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if not (ROOT / "run_video.py").is_file():
    raise RuntimeError("Open and run this notebook from the repository root or its notebooks directory.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from eval_track import load_ids, mot
from score import evaluate

OUTPUT_ROOT = ROOT / "preds" / "experiments"
LEGACY_ROOT = ROOT.parent / "CF Drone trecking "


def prefer(primary, fallback):
    return primary if primary.exists() or not fallback.exists() else fallback


DEFAULT_VIDEOS = prefer(ROOT.parent / "test_sample_clean", LEGACY_ROOT / "datasets" / "practice")
SKYGUARD_WEIGHTS = prefer(
    ROOT.parent / "pretrained" / "drone-detection-yolov11" / "best.pt",
    LEGACY_ROOT / "models" / "skyguard-best.pt",
)
RELEASE_WEIGHTS = ROOT / "runs" / "ft_1280" / "weights" / "epoch0.pt"


@dataclass(frozen=True)
class Experiment:
    name: str
    pipeline: str
    detector_weights: Path | str
    crop_weights: Path | str | None = None
    imgsz: int = 1280
    conf: float = 0.01
    keep_classes: str = "drone"
    device: str = ""
    tracking_options: dict = field(default_factory=dict)


@dataclass(frozen=True)
class Clip:
    name: str
    video: Path
    ground_truth: Path | None

## Configure the comparison

`VIDEO_SOURCE` accepts a local MP4 or a directory containing one folder per clip
with `video.mp4` and optional `gt/gt.txt`. Paths may be absolute or relative to the
repository. Set `VIDEO_NAMES` to clip folder names to use a subset. For a standalone
MP4, set `SINGLE_VIDEO_GT` if its labels are stored separately.

The default looks for the repository's documented sibling `test_sample_clean` folder,
then for the existing practice clips in `../CF Drone trecking /datasets/practice`.
The preflight cell shows the selected video and model paths before inference.

Each preset specifies a full-frame checkpoint, inference size, class filter,
confidence and device. The two crop presets select an independent crop checkpoint.
Use a single-class drone model for crops, since those runners process every crop
prediction. Clear or edit `keep_classes` if a full-frame model uses different labels.
Use local checkpoints from sources you trust.

`tracking_options` takes the existing script's argument names without the leading
dashes. The presets use `run_tracker.py`, `pipeline.py` and `pipeline_boost.py`.
Arguments set to `True` are enabled and those set to `False` are omitted.

In [ ]:
VIDEO_SOURCE = DEFAULT_VIDEOS
VIDEO_NAMES = []  # Empty selects all clips. Otherwise list exact folder names.
SINGLE_VIDEO_GT = None  # Optional path to MOT labels for a standalone MP4.
DEVICE = ""  # Auto. Set "cpu", "mps", or a CUDA device explicitly if needed.
MOT_MIN_CONF = 0.0

tracker_only = Experiment(
    "tracker-only", "tracker", SKYGUARD_WEIGHTS, device=DEVICE,
    tracking_options={"high": 0.6, "low": 0.05, "new": 0.3, "confirm": 3, "coast": 3},
)
trajectory = Experiment(
    "trajectory", "trajectory", SKYGUARD_WEIGHTS, crop_weights=SKYGUARD_WEIGHTS, device=DEVICE,
    tracking_options={"crop": 320, "crop-imgsz": 320, "no-motion": True},
)
release_boosted = Experiment(
    "v0.883-boosted", "boosted", SKYGUARD_WEIGHTS, crop_weights=RELEASE_WEIGHTS, device=DEVICE,
    tracking_options={
        "crop": 320, "crop-imgsz": 320, "crop-scale": 2, "no-motion": True,
        "layer-width": 480, "verify": True, "keeper2": True,
        "agree-boost": 0.1, "boost-to": 0.65,
    },
)

EXPERIMENTS = [tracker_only, trajectory, release_boosted]
# Example: compare a smaller detector by adding a fourth named experiment.
# EXPERIMENTS.append(replace(release_boosted, name="boosted-960", imgsz=960))
# For an existing ONNX export, use CPU and match the export's input size.
# EXPERIMENTS.append(replace(
#     tracker_only, name="onnx-640", device="cpu", imgsz=640,
#     detector_weights=LEGACY_ROOT / "models" / "skyguard-yolo11n.onnx",
# ))
# To swap models, set detector_weights and/or crop_weights with replace(...).

BATCH = None

In [ ]:
PIPELINES = {"tracker": "run_tracker.py", "trajectory": "pipeline.py", "boosted": "pipeline_boost.py"}
RESERVED_OPTIONS = {"dets", "out", "videos", "weights", "device", "imgsz"}


def local_path(value):
    return (ROOT / Path(value).expanduser()).resolve()


def discover_clips(source):
    if source.is_file() and source.suffix.lower() == ".mp4":
        name = source.parent.name if source.name == "video.mp4" else source.stem
        labels = local_path(SINGLE_VIDEO_GT) if SINGLE_VIDEO_GT is not None else source.parent / "gt" / "gt.txt"
        if SINGLE_VIDEO_GT is not None and not labels.is_file():
            raise ValueError(f"SINGLE_VIDEO_GT does not exist: {labels}")
        return [Clip(name.lstrip(".") or "clip", source, labels if labels.is_file() else None)]
    if SINGLE_VIDEO_GT is not None:
        raise ValueError("Set SINGLE_VIDEO_GT only with a single MP4 as VIDEO_SOURCE.")
    if not source.is_dir():
        raise ValueError(f"Set VIDEO_SOURCE to a local MP4 or dataset folder. Tried {source}")
    clips = []
    for video in sorted(source.glob("*/video.mp4")):
        if video.is_file() and not video.parent.name.startswith("."):
            labels = video.parent / "gt" / "gt.txt"
            clips.append(Clip(video.parent.name, video, labels if labels.is_file() else None))
    if not clips:
        raise ValueError(f"No <clip>/video.mp4 files found under {source}")
    return clips


def preflight():
    clips = discover_clips(local_path(VIDEO_SOURCE))
    if VIDEO_NAMES:
        missing = set(VIDEO_NAMES) - {clip.name for clip in clips}
        if missing:
            raise ValueError(f"Unknown VIDEO_NAMES: {', '.join(sorted(missing))}")
        clips = [clip for clip in clips if clip.name in VIDEO_NAMES]
    for clip in clips:
        capture = cv2.VideoCapture(str(clip.video))
        try:
            ok, _ = capture.read()
        finally:
            capture.release()
        if not ok:
            raise ValueError(f"OpenCV could not read the first frame of {clip.video}")

    if not EXPERIMENTS:
        raise ValueError("Add at least one experiment to EXPERIMENTS.")
    if not 0 <= MOT_MIN_CONF <= 1:
        raise ValueError("MOT_MIN_CONF must be between 0 and 1.")
    names = set()
    experiments = []
    for experiment in EXPERIMENTS:
        if not re.fullmatch(r"[A-Za-z0-9][A-Za-z0-9._-]*", experiment.name):
            raise ValueError(f"Use letters, digits, dots, dashes or underscores in {experiment.name!r}.")
        key = experiment.name.casefold()
        if key in names:
            raise ValueError(f"Duplicate experiment name: {experiment.name}")
        names.add(key)
        if experiment.pipeline not in PIPELINES:
            raise ValueError(f"Unknown pipeline {experiment.pipeline!r}. Choose from {list(PIPELINES)}.")
        reserved = RESERVED_OPTIONS.intersection(experiment.tracking_options)
        if reserved:
            raise ValueError(f"{experiment.name}: configure {sorted(reserved)} through Experiment fields.")
        if experiment.imgsz <= 0 or not 0 <= experiment.conf <= 1:
            raise ValueError(f"{experiment.name}: choose a positive imgsz and conf between 0 and 1.")
        detector = local_path(experiment.detector_weights)
        crop = None
        if experiment.pipeline != "tracker":
            if experiment.crop_weights is None:
                raise ValueError(f"{experiment.name}: set crop_weights.")
            crop = local_path(experiment.crop_weights)
        for role, weights in (("detector", detector), ("crop", crop)):
            if weights is not None and not weights.is_file():
                raise ValueError(f"{experiment.name}: {role} weights not found at {weights}")
        experiments.append(replace(experiment, detector_weights=detector, crop_weights=crop))
    return clips, experiments


try:
    CLIPS, RUN_CONFIGS = preflight()
except ValueError as error:
    CLIPS, RUN_CONFIGS = [], []
    print(f"Update the configuration to continue. {error}")
else:
    print(f"Video source: {local_path(VIDEO_SOURCE)}")
    display(pd.DataFrame([{"clip": clip.name, "labeled": clip.ground_truth is not None} for clip in CLIPS]))
    display(pd.DataFrame([
        {
            "experiment": cfg.name, "pipeline": cfg.pipeline, "imgsz": cfg.imgsz, "conf": cfg.conf,
            "device": cfg.device or "auto", "full frame model": str(cfg.detector_weights),
            "crop model": str(cfg.crop_weights) if cfg.crop_weights else "",
        }
        for cfg in RUN_CONFIGS
    ]))

## Run detection and tracking

The notebook writes a new batch under `preds/experiments/` on every run. Matching
full-frame detector settings share detections within that batch. Each experiment
keeps its own tracks and command log, and each batch records the selected inputs
and options in `settings.json`.

The tracker-only preset reads video frames for appearance re-identification.
Trajectory and boosted presets run the repository's crop pipelines with their
built-in tracker settings. The release preset uses the downloaded v0.883 checkpoint
on trajectory crops and enables the consensus verifier, booster and motion-consistent keeper.

In [ ]:
def cli_options(options):
    args = []
    for name, value in options.items():
        if value is False or value is None:
            continue
        args.append(f"--{name}")
        if value is not True:
            args.append(str(value))
    return args


def run_stage(script, args, log):
    command = [sys.executable, "-u", str(ROOT / script), *map(str, args)]
    print(f"Running {script} → {log.relative_to(ROOT)}", flush=True)
    start = perf_counter()
    with log.open("w", encoding="utf-8") as output:
        output.write(shlex.join(command) + "\n\n")
        with subprocess.Popen(
            command, cwd=ROOT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, errors="replace", bufsize=1,
        ) as process:
            try:
                for line in process.stdout:
                    output.write(line)
                    print(line, end="", flush=True)
                code = process.wait()
            except KeyboardInterrupt:
                process.terminate()
                raise
    if code:
        raise RuntimeError(f"{script} exited with status {code}. See {log}")
    return perf_counter() - start


def check_outputs(folder, clips):
    missing = [clip.name for clip in clips if not (folder / f"{clip.name}.txt").is_file()]
    if missing:
        raise RuntimeError(f"Missing output in {folder} for {', '.join(missing)}")


def run_batch(clips, experiments):
    OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
    prefix = f"{datetime.now():%Y%m%d-%H%M%S}-"
    folder = Path(tempfile.mkdtemp(prefix=prefix, dir=OUTPUT_ROOT))
    settings = {"clips": list(map(asdict, clips)), "experiments": list(map(asdict, experiments)),
                "mot_min_conf": MOT_MIN_CONF, "python": sys.executable}
    (folder / "settings.json").write_text(json.dumps(settings, default=str, indent=2), encoding="utf-8")

    # The CLI scripts read <clip>/video.mp4. Link only the chosen videos into this layout.
    video_root = folder / "videos"
    for clip in clips:
        directory = video_root / clip.name
        directory.mkdir(parents=True)
        (directory / "video.mp4").symlink_to(clip.video)

    detector_cache = {}
    results = {}
    for experiment in experiments:
        run_dir = folder / "configs" / experiment.name
        run_dir.mkdir(parents=True)
        print(f"\n{experiment.name}", flush=True)
        detector_key = (experiment.detector_weights, experiment.imgsz, experiment.conf,
                        experiment.keep_classes, experiment.device)
        reused = detector_key in detector_cache
        device = cli_options({"device": experiment.device or None})
        if reused:
            detections, detection_seconds = detector_cache[detector_key]
            print(f"Reusing detections from {detections.relative_to(folder)}", flush=True)
        else:
            detections = run_dir / "detections"
            detection_seconds = run_stage("run_video.py", [
                "--weights", experiment.detector_weights, "--videos", video_root, "--out", detections,
                "--imgsz", experiment.imgsz, "--conf", experiment.conf,
                *cli_options({"keep": experiment.keep_classes or None}), *device,
            ], run_dir / "detect.log")
            check_outputs(detections, clips)
            detector_cache[detector_key] = (detections, detection_seconds)

        tracks = run_dir / "tracks"
        tracking_args = ["--dets", detections, "--videos", video_root, "--out", tracks]
        if experiment.crop_weights:
            tracking_args.extend(["--weights", experiment.crop_weights, *device])
        tracking_seconds = run_stage(
            PIPELINES[experiment.pipeline],
            [*tracking_args, *cli_options(experiment.tracking_options)],
            run_dir / "track.log",
        )
        check_outputs(tracks, clips)
        results[experiment.name] = {
            "detections": detections, "tracks": tracks, "detection_s": detection_seconds,
            "tracking_s": tracking_seconds, "detections_reused": reused,
        }
    return {"folder": folder, "clips": clips, "results": results, "mot_min_conf": MOT_MIN_CONF}

In [ ]:
BATCH = None
if CLIPS and RUN_CONFIGS:
    BATCH = run_batch(CLIPS, RUN_CONFIGS)
    print(f"\nBatch saved to {BATCH['folder']}")
else:
    print("Resolve the input check above and rerun the configuration and preflight cells.")

## Compare experiment results

The stage times include model startup, video decode and writing predictions.
Each experiment's combined time includes the full detection cost, even when the
batch reused those detections. Lower times indicate shorter processing.

MOT scores use IoU ≥ 0.5 on labeled clips only. The MOT confidence threshold applies
to tracking metrics. AP50 uses all prediction scores, and its overall value is
the mean of labeled clips. Track counts and overlays are also available for
unlabeled video.

In [ ]:
MOT_COLUMNS = ["mota", "idf1", "precision", "recall", "num_switches", "num_false_positives", "num_misses"]


def evaluate_tracks(clips, tracks, min_conf):
    accumulators = []
    ap50 = []
    for clip in clips:
        truth = load_ids(clip.ground_truth, False)
        predictions = load_ids(tracks / f"{clip.name}.txt", True)
        accumulators.append(mot(truth, predictions, min_conf))
        # Reuse the repository's AP evaluator, which takes boxes without track IDs.
        ap50.append(evaluate(
            {frame: [box[:5] for box in boxes] for frame, boxes in truth.items()},
            {frame: [box[:5] for box in boxes] for frame, boxes in predictions.items()},
            0.5,
        )[0])
    metrics = mm.metrics.create().compute_many(
        accumulators, generate_overall=True, metrics=MOT_COLUMNS,
    )
    metrics.index = pd.Index([*(clip.name for clip in clips), "OVERALL"], name="clip")
    metrics["AP50"] = [*ap50, np.mean(ap50)]
    return metrics


def compare(batch):
    stage_rows = []
    track_rows = []
    scores = {}
    labeled = [clip for clip in batch["clips"] if clip.ground_truth is not None]
    for name, result in batch["results"].items():
        stage_rows.append({
            "experiment": name, "detection_s": result["detection_s"],
            "tracking_s": result["tracking_s"],
            "combined_s": result["detection_s"] + result["tracking_s"],
            "detections_reused": result["detections_reused"],
        })
        for clip in batch["clips"]:
            raw = load_ids(result["detections"] / f"{clip.name}.txt", True)
            tracked = load_ids(result["tracks"] / f"{clip.name}.txt", True)
            track_rows.append({
                "experiment": name, "clip": clip.name,
                "detected_boxes": sum(map(len, raw.values())),
                "tracked_boxes": sum(map(len, tracked.values())),
                "frames_with_tracks": len(tracked),
            })
        if labeled:
            scores[name] = evaluate_tracks(labeled, result["tracks"], batch["mot_min_conf"])

    folder = batch["folder"]
    stage_times = pd.DataFrame(stage_rows).set_index("experiment")
    counts = pd.DataFrame(track_rows).set_index(["experiment", "clip"])
    stage_times.to_csv(folder / "timings.csv")
    counts.to_csv(folder / "track_counts.csv")
    display(stage_times.round(2), counts)
    if not scores:
        print("No ground-truth files were found for the selected clips.")
        return
    metrics = pd.concat(scores, names=["experiment", "clip"])
    overall = pd.DataFrame({name: score.iloc[-1] for name, score in scores.items()}).T
    overall.index.name = "experiment"
    metrics.to_csv(folder / "mot_metrics.csv")
    print("Overall scores")
    display(overall.round(3))
    print("Per-clip scores")
    display(pd.concat({name: score.iloc[:-1] for name, score in scores.items()}).round(3))


if BATCH:
    compare(BATCH)

## Inspect an annotated frame

Leave the preview settings empty to use the first clip and a representative
frame with labels or predictions. Set a clip name and a 1-based frame number to
inspect the same frame across all experiments. Cyan boxes show ground truth and
green boxes show tracks, with each track ID and prediction confidence.

In [ ]:
PREVIEW_CLIP = None
PREVIEW_FRAME = None


def draw_boxes(frame, boxes, color, *, truth=False):
    height, width = frame.shape[:2]
    for x1, y1, x2, y2, conf, track_id in boxes:
        left, right = (min(max(round(x), 0), width - 1) for x in (x1, x2))
        top, bottom = (min(max(round(y), 0), height - 1) for y in (y1, y2))
        text = f"GT {track_id}" if truth else f"ID {track_id} ({conf:.2f})"
        y = min(bottom + 17, height - 4) if truth else max(top - 5, 16)
        cv2.rectangle(frame, (left, top), (right, bottom), color, 2)
        cv2.putText(frame, text, (left, y), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (0, 0, 0), 3)
        cv2.putText(frame, text, (left, y), cv2.FONT_HERSHEY_SIMPLEX, 0.55, color, 1)


def preview(batch, clip_name=None, frame_number=None):
    clips = {clip.name: clip for clip in batch["clips"]}
    if clip_name is not None and clip_name not in clips:
        raise ValueError(f"Choose PREVIEW_CLIP from {list(clips)}.")
    clip = clips[clip_name] if clip_name is not None else next(iter(clips.values()))
    truth = load_ids(clip.ground_truth, False) if clip.ground_truth is not None else {}
    predictions = {
        name: load_ids(result["tracks"] / f"{clip.name}.txt", True)
        for name, result in batch["results"].items()
    }
    if frame_number is None:
        frames = sorted(truth or set().union(*(set(track) for track in predictions.values())))
        frame_number = frames[len(frames) // 2] if frames else 1
    if type(frame_number) is not int or frame_number < 1:
        raise ValueError("PREVIEW_FRAME must be a positive, 1-based integer.")

    capture = cv2.VideoCapture(str(clip.video))
    try:
        capture.set(cv2.CAP_PROP_POS_FRAMES, frame_number - 1)
        ok, image = capture.read()
    finally:
        capture.release()
    if not ok:
        raise ValueError(f"Could not read frame {frame_number} from {clip.video}.")

    print(f"{clip.name}, frame {frame_number}")
    for name, tracks in predictions.items():
        panel = image.copy()
        draw_boxes(panel, tracks.get(frame_number, []), (80, 235, 80))
        draw_boxes(panel, truth.get(frame_number, []), (255, 235, 60), truth=True)
        if panel.shape[1] > 1000:
            scale = 1000 / panel.shape[1]
            panel = cv2.resize(panel, None, fx=scale, fy=scale, interpolation=cv2.INTER_AREA)
        ok, jpeg = cv2.imencode(".jpg", panel)
        if not ok:
            raise RuntimeError("OpenCV could not encode the preview frame.")
        print(name)
        display(Image(data=jpeg.tobytes(), format="jpeg"))


if BATCH:
    preview(BATCH, PREVIEW_CLIP, PREVIEW_FRAME)